# Assignment 6 — Neural Machine Translation

> **Instructor**: Dr. Ye Kyaw Thu

> **Class**: AIE-F (Batch-2)

> **Submission**: Thant Sin Tun

> **Date**: 3/10/2026

## Myanmar ↔ Hiero Translation using Marian NMT

This project implements Neural Machine Translation between Myanmar and Hiero using the Marian NMT framework.

The experiments include:

1. Data preprocessing
2. Train / validation / test splitting
3. Vocabulary construction (syllable & word unit)
4. Seq2Seq NMT
5. Transformer NMT
6. Myanmar → Hiero translation
7. Hiero → Myanmar translation
8. Hyperparameter tuning
9. Ensemble Experiment (MY-HI, HI-MY)
10. Qualitative translation analysis
11. Model comparison

Reference notebooks:

- ALT-Corpus-Translation-Tutorial.ipynb
- NMT-Tutorial-with-myContradict.ipynb
- Marian Framework: [https://github.com/marian-nmt/marian](https://github.com/marian-nmt/marian)  

## Project Directory

In [2]:
%cd ~
%cd exp/nmt/assignment-6

/home/thant_syn
/home/thant_syn/exp/nmt/assignment-6


In [3]:
%pwd

'/home/thant_syn/exp/nmt/assignment-6'

In [4]:
%ls

burmese_hiero_mt.ipynb  myhi_data/


In [5]:
!tree 

.
├── burmese_hiero_mt.ipynb
└── myhi_data
    ├── test.hi
    ├── test.my
    ├── train.hi
    ├── train.my
    ├── valid.hi
    └── valid.my

2 directories, 7 files


## Check Data Alignment

In [6]:
%cd myhi_data/

/home/thant_syn/exp/nmt/assignment-6/myhi_data


In [7]:
!wc -l *.my

   500 test.my
  8500 train.my
  1000 valid.my
 10000 total


In [8]:
!wc -l *.hi

   500 test.hi
  8500 train.hi
  1000 valid.hi
 10000 total


In [9]:
# parallel check
print("Train")
!paste train.my train.hi | head

print("\nTest")
!paste test.my test.hi | head

print('\n')
!echo "Valid"
!paste valid.my valid.hi | head

Train
မင်းက အေးဆေးတယ်	𓂨𓀀𓎡𓋴𓂝 𓏮𓀀𓐍𓂋
ရိုးသားမှုရှိတယ်	𓏏𓂝𓋴𓂝 𓈙𓆑𓎛𓏭𓂝𓋴𓂝
ကျွန်မ ရိုးသားနိုင်လား။	𓏭𓄿𓎛 𓆑 𓇋𓂝 𓊪𓐍𓎛𓂝𓈙𓈎
ဘယ်လို သက်သာမှုလဲ။	𓏏𓊪𓄿𓈎 𓄿 𓋴𓂝𓂋𓆑𓂝𓅱
ကျွန်မသွားခွင့်ပြုပါ။	𓄿𓂋𓂋𓐍𓏏 𓉔𓂝 𓈎𓐍 𓃀𓐍
ကျွန်တော်တို့ မြင်ခဲ့တယ်။	𓏏𓂝 𓈙𓄿𓏏 𓆑𓈎
ကိုယ့်ကိုယ်ကို လေးစားပါ။	𓋴𓂝𓈙𓄡𓂝𓏭𓈎 𓂨𓀀𓎡𓋴𓈙𓂝𓂋𓅱
သန့်ရှင်းလား။	𓆑𓈙 𓆑𓈎 𓏭𓂋𓂝𓄿𓎛
ငါးတစ်ကောင်က ရေကူးနိုင်တယ်	𓄿 𓅱𓆑𓈙𓊪 𓏭𓄿𓎛 𓈙𓏏𓆑𓉔
ဆေးလိပ်သောက်တာ ရပ်လိုက်ပါ	𓈙𓈎𓐍𓄡 𓈙𓉔𓐍𓈖𓆑𓎛𓃀
paste: write error: Broken pipe

Test
အချစ်က နာကျင်စေတယ်။	𓂋𓐍𓎼𓂝 𓊪𓎡𓋴𓈎𓈙
ကျွန်မမှာ အစားအစာမရှိဘူး။	𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌
ဘယ်သူမှ ကျွန်မကို မမြင်ခဲ့ဘူး။	𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝
ခင်ဗျားက သတိထားတတ်သူပါ။	𓂨𓀀𓎡𓋴𓂝 𓐍𓇋𓈙𓂝𓋴𓎼𓄿𓎛𓈎
အသင်းက စောင့်နေခဲ့တယ်။	𓈎𓊪𓂝 𓈎𓂝𓄿𓉔 𓏏𓄿𓆑𓈎𓂝𓇌
ခရီးကောင်းပါ	𓊪𓄿𓎼𓂝 𓄿 𓎛𓆑𓏭𓂝 𓈎𓋴𓆑𓄡
ဘယ်လိုပဲဖြစ်ဖြစ် လုပ်ခဲ့တယ်။	𓆑 𓇌𓆑𓇌 𓆑𓈎 𓄿𓎛𓂧𓏏𓄿𓂧
အခန်းက မှောင်နေတယ်။	𓈎𓊪𓂝 𓋴𓐍𓐍𓉔 𓆑𓈙 𓇌𓄿𓋴𓈖
သူမ လမ်းလျှောက်တယ်။	𓈙𓊪𓂝 𓏏𓄿𓂋𓈖𓈙
ဒါပဲဆို	𓈎𓊪𓄿𓈎𓈙 𓄿𓇋𓐍𓎡𓈎 𓄿𓂋𓂋
paste: write error: Broken pipe


Valid
ကျွန်မ အဆင်ပြေပါတယ်။	𓆑 𓅱𓂝𓂝𓂋 𓏏𓂝𓂋𓂋
ကျွန်မမှာ မြင်းမရှိဘူး။	𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓊪𓐍𓋴𓈙𓂝
သူ့ကို ငိုစေခဲ့တယ်။	𓏏𓂝 𓉔𓄿𓇌𓂝 𓊪𓆑𓉔 𓏭𓋴𓂧
နည်းနည်း ပိုနီးလာပါဦး။	𓏮𓀀𓉔𓂝 𓄿 𓇋𓆑𓈎 𓏭𓂋𓐍𓈙𓂝𓋴
ကျွန်တော် ယုံကြည်ပါတယ်။	𓆑𓉔 𓏮𓀀𓎛𓅱𓆑𓇌𓂝𓎛𓈎
ချိုတာကြိုက်တယ

### Environment Setup

In [1]:
import os

os.environ["PATH"] = "/home/thant_syn/marian-dev/build:" + os.environ["PATH"]

In [2]:
!marian --help

Marian: Fast Neural Machine Translation in C++
Usage: marian [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=2048               Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-time-zone TEXT  

In [3]:
!which marian

/home/thant_syn/marian-dev/build/marian


In [4]:
!marian --version

v1.12.43 c9f287db 2025-06-23 19:47:45 +0000


In [5]:
!marian --help | head -n 30

Marian: Fast Neural Machine Translation in C++
Usage: marian [OPTIONS]

General options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  --authors                             Print list of authors and exit
  --cite                                Print citation and exit
  --build-info TEXT                     Print CMake build options and exit. Set to 'all' to print advanced options
  -c,--config VECTOR ...                Configuration file(s). If multiple, later overrides earlier
  -w,--workspace INT=2048               Preallocate arg MB of work space. Negative `--workspace -N` value allocates workspace as total available GPU memory minus N megabytes.
  --log TEXT                            Log training process information to file given by arg
  --log-level TEXT=info                 Set verbosity level of logging: trace, debug, info, warn, err(or), critical, off
  --log-time-zone TEXT  

In [6]:
!marian-vocab --help

Create a vocabulary from text corpora given on STDIN
Usage: marian-vocab [OPTIONS]

Allowed options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  -m,--max-size UINT=0                  Generate only UINT most common vocabulary items

Examples:
  ./marian-vocab < text.src > vocab.yml
  cat text.src text.trg | ./marian-vocab > vocab.yml


## Data Directory

In [6]:
DATA_DIR = os.path.expanduser("~/exp/nmt/assignment-6/myhi_data")
print("Dataset directory:", DATA_DIR)

Dataset directory: /home/thant_syn/exp/nmt/assignment-6/myhi_data


In [7]:
!pwd

/home/thant_syn/exp/nmt/assignment-6


In [8]:
!find ~/exp/nmt/assignment-6/myhi_data -maxdepth 2 -type f

/home/thant_syn/exp/nmt/assignment-6/myhi_data/test.my
/home/thant_syn/exp/nmt/assignment-6/myhi_data/train.hi
/home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.hi
/home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.my
/home/thant_syn/exp/nmt/assignment-6/myhi_data/test.hi
/home/thant_syn/exp/nmt/assignment-6/myhi_data/train.my


### Inspect Data

In [9]:
import os

for root, dirs, files in os.walk(DATA_DIR):
    level = root.replace(DATA_DIR, "").count(os.sep)
    indent = "    " * level
    print(f"{indent}{os.path.basename(root)}/")

    subindent = "    " * (level + 1)

    for file in files:
        print(f"{subindent}{file}")

myhi_data/
    test.my
    train.hi
    valid.hi
    valid.my
    test.hi
    train.my
    assignment-6/
        scripts/
        results/
        models/
        vocab/
        data/


In [11]:
!find /home/thant_syn/exp/nmt/assignment-6/myhi_data -type f -exec wc -l {} \;

500 /home/thant_syn/exp/nmt/assignment-6/myhi_data/test.my
8500 /home/thant_syn/exp/nmt/assignment-6/myhi_data/train.hi
1000 /home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.hi
1000 /home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.my
500 /home/thant_syn/exp/nmt/assignment-6/myhi_data/test.hi
8500 /home/thant_syn/exp/nmt/assignment-6/myhi_data/train.my


In [15]:
from pathlib import Path

source_file = Path("/home/thant_syn/exp/nmt/assignment-6/myhi_data/train.my")
target_file = Path("/home/thant_syn/exp/nmt/assignment-6/myhi_data/train.hi")

with open(source_file, "r", encoding="utf-8") as f:
    source_lines = f.readlines()

with open(target_file, "r", encoding="utf-8") as f:
    target_lines = f.readlines()

print("Source sentences:", len(source_lines))
print("Target sentences:", len(target_lines))

assert len(source_lines) == len(target_lines), \
    "Source and target sentence counts do not match!"

print("\n✓ Source and target are aligned.")

Source sentences: 8500
Target sentences: 8500

✓ Source and target are aligned.


## Tokenization/Segmentation for Myanmar Language

In [34]:
%cd ~/exp/nmt

/home/thant_syn/exp/nmt


In [18]:
!git clone https://github.com/ye-kyaw-thu/oppaWord

Cloning into 'oppaWord'...
remote: Enumerating objects: 449, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 449 (delta 44), reused 0 (delta 0), pack-reused 374 (from 1)
Receiving objects: 100% (449/449), 288.83 MiB | 1.65 MiB/s, done.
Resolving deltas: 100% (242/242), done.
Updating files: 100% (199/199), done.


In [35]:
%pwd

'/home/thant_syn/exp/nmt'

In [38]:
%cd oppaWord/

/home/thant_syn/exp/nmt/oppaWord


In [39]:
%ls tools

correct_my_punc.py    evaluate.py  modify_tex.py       run_experiment.sh*
eval_segmentation.py  mk_id.pl     print-codepoint.pl  smart_space_remover.py


In [50]:
!python oppa_word.py --help

usage: oppa_word.py [-h] --input INPUT [--output OUTPUT] --dict DICT
                    [--sylfreq SYLFREQ] [--arpa ARPA]
                    [--postrule-file POSTRULE_FILE] [--max-order MAX_ORDER]
                    [--dict-weight DICT_WEIGHT] [--use-bimm-fallback]
                    [--bimm-boost BIMM_BOOST] [--visualize-dag]
                    [--dag-output-dir DAG_OUTPUT_DIR]
                    [--space-remove-mode {all,my,my_not_num}]
                    [--max-word-len MAX_WORD_LEN]

oppa_word, Hybrid DAG + BiMM + LM Myanmar Word Segmenter with optional Aho-
Corasick support

options:
  -h, --help            show this help message and exit
  --input INPUT, -i INPUT
                        Input file with one sentence per line (UTF-8)
  --output OUTPUT, -o OUTPUT
                        Optional output file path (default: stdout)
  --dict DICT, -d DICT  Word dictionary file (one word per line)
  --sylfreq SYLFREQ, -s SYLFREQ
                        Syllable frequency file (sy

In [25]:
# !pip install kenlm

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for kenlm: filename=kenlm-0.3.0-cp312-cp312-linux_x86_64.whl size=3164113 sha256=af89e9d2070943b4a2ebaec274e4a851e0bb445c73da9cc5f284f40dce82d9cd
  Stored in directory: /home/thant_syn/.cache/pip/wheels/0c/e6/ad/18d2d3f1290a6be6a14a24e90f2b78bb3300aab3852ceb06a6
Successfully built kenlm


In [42]:
!time python oppa_word.py \
  --input /home/thant_syn/exp/nmt/assignment-6/myhi_data/train.my \
  --dict data/myg2p_mypos.dict \
  --use-bimm-fallback \
  --bimm-boost 150 \
  --space-remove-mode "my_not_num" \
  > /home/thant_syn/exp/nmt/assignment-6/myhi_data/word_train.my


real	0m0.592s
user	0m0.574s
sys	0m0.044s


In [45]:
!head -n 15 /home/thant_syn/exp/nmt/assignment-6/myhi_data/word_train.my

မင်း က အေးဆေး တယ်
ရိုးသားမှု ရှိတယ်
ကျွန်မ ရိုးသား နိုင် လား ။
ဘယ်လို သက်သာ မှု လဲ ။
ကျွန်မ သွား ခွင့်ပြု ပါ ။
ကျွန်တော် တို့ မြင် ခဲ့ တယ် ။
ကိုယ့် ကိုယ်ကို လေးစား ပါ ။
သန့်ရှင်း လား ။
ငါး တစ် ကောင် က ရေကူး နိုင် တယ်
ဆေးလိပ် သောက် တာ ရပ် လိုက်ပါ
စကား မ ပြော ပါ နဲ့ ။
Tom ဘယ်တော့ မှ မ အိပ် ဘူး ။
ကျန်းမာ နေ တယ် လို့ ခံစား ရ တယ် ။
ကျဉ်းမြောင်း တယ် ။
တီဗီ ကြည့် တာ မ ကြည့် ဘူး ။


In [46]:
!time python oppa_word.py \
  --input /home/thant_syn/exp/nmt/assignment-6/myhi_data/test.my \
  --dict data/myg2p_mypos.dict \
  --use-bimm-fallback \
  --bimm-boost 150 \
  --space-remove-mode "my_not_num" \
  > /home/thant_syn/exp/nmt/assignment-6/myhi_data/word_test.my


real	0m0.093s
user	0m0.070s
sys	0m0.026s


In [47]:
!head -n 15 /home/thant_syn/exp/nmt/assignment-6/myhi_data/word_test.my

အချစ် က နာကျင် စေ တယ် ။
ကျွန်မ မှာ အစားအစာ မရှိ ဘူး ။
ဘယ်သူ မှ ကျွန်မ ကို မ မြင် ခဲ့ ဘူး ။
ခင်ဗျား က သတိထား တတ် သူ ပါ ။
အသင်း က စောင့် နေ ခဲ့ တယ် ။
ခရီး ကောင်း ပါ
ဘယ်လို ပဲ ဖြစ် ဖြစ် လုပ် ခဲ့ တယ် ။
အခန်း က မှောင် နေ တယ် ။
သူမ လမ်းလျှောက် တယ် ။
ဒါပဲ ဆို
နင့် နံ့ က ကောင်း တယ်
ကျွန်တော် ပြီးအောင် ပြော ပါရစေ ။
ကျွန်မ တာဝန် ရှိ နေ တယ် ။
ပို ဆိုး တာ ကို မျှော်လင့် ခဲ့ တယ် ။
မင်း ကို ကူညီ ပါရစေ ။


In [48]:
!time python oppa_word.py \
  --input /home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.my \
  --dict data/myg2p_mypos.dict \
  --use-bimm-fallback \
  --bimm-boost 150 \
  --space-remove-mode "my_not_num" \
  > /home/thant_syn/exp/nmt/assignment-6/myhi_data/word_valid.my


real	0m0.107s
user	0m0.096s
sys	0m0.013s


In [49]:
!head -n 15 /home/thant_syn/exp/nmt/assignment-6/myhi_data/word_valid.my

ကျွန်မ အဆင်ပြေ ပါတယ် ။
ကျွန်မ မှာ မြင်းမ ရှိ ဘူး ။
သူ့ ကို ငို စေ ခဲ့ တယ် ။
နည်းနည်း ပို နီး လာ ပါဦး ။
ကျွန်တော် ယုံကြည် ပါတယ် ။
ချို တာ ကြိုက် တယ် ။
စိတ်မကောင်း ပါ ဘူး ။
ဒါ က အရသာ ကောင်း တယ် ။
ပို ကောင်း လာ ပြီ ။
ကျွန်မ က ယာ ဗီ ယာ စား ခဲ့ တယ် ။
ဒါ က အများကြီး ပါ ။
အားလုံး ပြီး သွား ပြီ ။
သူ ပြေး နေ တယ် ။
ပြီး သွား ပြီ ။
သူမ က သင့် ကို ယုံကြည် ခဲ့ တယ် ။


## Syllable Segmentation

In [51]:
%pwd

'/home/thant_syn/exp/nmt/oppaWord'

In [52]:
%cd ..

/home/thant_syn/exp/nmt


In [53]:
%mkdir sylbreak

In [54]:
%cd sylbreak/

/home/thant_syn/exp/nmt/sylbreak


In [55]:
!wget https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak4all/refs/heads/main/python/sylbreak4all.py

--2026-10-03 06:18:58--  https://raw.githubusercontent.com/ye-kyaw-thu/sylbreak4all/refs/heads/main/python/sylbreak4all.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.111.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4796 (4.7K) [text/plain]
Saving to: ‘sylbreak4all.py’

sylbreak4all.py     100%[===================>]   4.68K  --.-KB/s    in 0.001s  

2026-10-03 06:18:59 (6.62 MB/s) - ‘sylbreak4all.py’ saved [4796/4796]



In [56]:
!python sylbreak4all.py --help

Syllable Breaker Tool
Version: 0.9.0

Supported Languages:
    bm  - Burmese
    bk  - Beik
    dw  - Dawei
    rk  - Rakhine
    mo  - Mon
    po  - PaO
    sh  - Shan
    sk  - Sgaw Kayin
    pk  - Pwo Kayin

Usage: sylbreak4all [-i filename] [-l language] [-s separator] [-p]
    -i | --input-file   Input file name (default: STDIN)
    -s | --separator    Separator (default: '|')
    -l | --language     Language (default: bm for Burmese)
    -p | --print        Print input alongside the output (default: 0)
    -h | --help         Show this help


In [57]:
%pwd

'/home/thant_syn/exp/nmt/sylbreak'

In [58]:
%cd ..

/home/thant_syn/exp/nmt


In [59]:
%cd assignment-6/myhi_data/

/home/thant_syn/exp/nmt/assignment-6/myhi_data


In [61]:
%mkdir syllable_segmented

In [64]:
!time python /home/thant_syn/exp/nmt/sylbreak/sylbreak4all.py -i /home/thant_syn/exp/nmt/assignment-6/myhi_data/train.my -l bm -s " " > /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_train.my


real	0m0.100s
user	0m0.088s
sys	0m0.013s


In [65]:
!head -n 10 /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_train.my

မင်း က အေး ဆေး တယ်
ရိုး သား မှု ရှိ တယ်
ကျွန် မ ရိုး သား နိုင် လား ။
ဘယ် လို သက် သာ မှု လဲ ။
ကျွန် မ သွား ခွ င့် ပြု ပါ ။
ကျွန် တော် တို့ မြင် ခဲ့ တယ် ။
ကို ယ့် ကိုယ် ကို လေး စား ပါ ။
သ န့် ရှင်း လား ။
ငါး တစ် ကောင် က ရေ ကူး နိုင် တယ်
ဆေး လိပ် သောက် တာ ရပ် လိုက် ပါ


In [66]:
!time python /home/thant_syn/exp/nmt/sylbreak/sylbreak4all.py -i /home/thant_syn/exp/nmt/assignment-6/myhi_data/test.my -l bm -s " " > /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_test.my


real	0m0.044s
user	0m0.038s
sys	0m0.008s


In [67]:
!head -n 10 /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_test.my

အ ချစ် က နာ ကျင် စေ တယ် ။
ကျွန် မ မှာ အ စား အ စာ မ ရှိ ဘူး ။
ဘယ် သူ မှ ကျွန် မ ကို မ မြင် ခဲ့ ဘူး ။
ခင် ဗျား က သ တိ ထား တတ် သူ ပါ ။
အ သင်း က စော င့် နေ ခဲ့ တယ် ။
ခ ရီး ကောင်း ပါ
ဘယ် လို ပဲ ဖြစ် ဖြစ် လုပ် ခဲ့ တယ် ။
အ ခန်း က မှောင် နေ တယ် ။
သူ မ လမ်း လျှောက် တယ် ။
ဒါ ပဲ ဆို


In [68]:
!time python /home/thant_syn/exp/nmt/sylbreak/sylbreak4all.py -i /home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.my -l bm -s " " > /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_valid.my


real	0m0.050s
user	0m0.043s
sys	0m0.009s


In [69]:
!head -n 10 /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_valid.my

ကျွန် မ အ ဆင် ပြေ ပါ တယ် ။
ကျွန် မ မှာ မြင်း မ ရှိ ဘူး ။
သူ့ ကို ငို စေ ခဲ့ တယ် ။
နည်း နည်း ပို နီး လာ ပါ ဦး ။
ကျွန် တော် ယုံ ကြည် ပါ တယ် ။
ချို တာ ကြိုက် တယ် ။
စိတ် မ ကောင်း ပါ ဘူး ။
ဒါ က အ ရ သာ ကောင်း တယ် ။
ပို ကောင်း လာ ပြီ ။
ကျွန် မ က ယာ ဗီ ယာ စား ခဲ့ တယ် ။


## Vocab Building


Neural Network ကို သုံးမှာမို့ ထုံးစံအတိုင်း vocab ဖိုင်တွေကို ပြင်ဆင်ကြရပါတယ်။  

ဒေတာထဲက syllable unit, word unit ဖြတ်ထားတဲ့ နှစ်မျိုးစလုံးအတွက် vocab ဆောက်ကြရအောင်။ 

ဘာကြောင့်လဲ ဆိုတော့ ဒီနေ့ syllable နဲ့ရော word နဲ့ရော အနည်းဆုံး NMT model ဆောက်ပြီး ရလဒ်တွေကို နှိုင်းယှဉ်ကြည့်ချင်လို့။  

In [72]:
!marian-vocab --help

Create a vocabulary from text corpora given on STDIN
Usage: marian-vocab [OPTIONS]

Allowed options:
  -h,--help                             Print this help message and exit
  --version                             Print the version number and exit
  -m,--max-size UINT=0                  Generate only UINT most common vocabulary items

Examples:
  ./marian-vocab < text.src > vocab.yml
  cat text.src text.trg | ./marian-vocab > vocab.yml


### For Syllable Segmentation

In [73]:
%cd ~/exp/nmt/assignment-6/myhi_data/syllable_segmented/

/home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented


In [74]:
!time cat syl_test.my  ../test.hi  syl_train.my  ../train.hi  syl_valid.my  ../valid.hi | marian-vocab > vocab.syl.yml

[2026-10-03 06:33:14] Creating vocabulary...
[2026-10-03 06:33:14] [data] Creating vocabulary stdout from stdin
[2026-10-03 06:33:14] Finished

real	0m0.189s
user	0m0.502s
sys	0m0.830s


In [75]:
!head -n 20 ./vocab.syl.yml

</s>: 0
<unk>: 1
။: 2
တယ်: 3
မ: 4
ပါ: 5
က: 6
𓆑: 7
ကျွန်: 8
ကို: 9
အ: 10
သူ: 11
ခဲ့: 12
နေ: 13
ဒါ: 14
ဘူး: 15
𓆑𓈙: 16
𓄿: 17
𓂨𓀀𓎡: 18
လား: 19


In [76]:
!tail -n 20 ./vocab.syl.yml

𓐍𓇋𓂝𓈙𓂝: 4146
𓐍𓇋𓅓𓂝𓏭𓈎: 4147
𓐍𓇋𓈚𓀀𓂋𓂝𓈎𓂝: 4148
𓐍𓇋𓎜𓀀𓍿𓆒𓀀𓎡𓈙: 4149
𓐍𓋴𓄿𓎛𓃀𓂝𓈙: 4150
𓐍𓋴𓏭𓊪𓆑𓇌𓈙: 4151
𓐍𓎡𓈎𓂋𓄿𓏏: 4152
𓐍𓎡𓈎𓇌𓄿𓈎𓂝𓇌: 4153
𓐍𓎡𓈎𓋴𓄿𓃀𓂝: 4154
𓐍𓎡𓈎𓋴𓄿𓎛𓈖: 4155
𓐍𓎡𓈎𓏏𓆑𓈎𓈎𓂝𓇌: 4156
𓐍𓎼𓂝𓋴𓇌𓐍: 4157
𓐍𓎼𓂝𓋴𓋴𓂝𓄿𓏭𓈎𓆑𓎛𓃀: 4158
𓐍𓎼𓂝𓋴𓏏𓂝𓆑𓃀𓊪𓈎: 4159
𓐍𓎼𓂝𓎛: 4160
𓐍𓏏𓂋: 4161
𓐍𓏏𓂝𓈙: 4162
𓐍𓏏𓎛𓂝𓋴: 4163
𓐍𓏏𓎛𓂝𓋴𓈙: 4164
𓐍𓏭𓏭𓎡𓄡𓆑𓂝𓇌: 4165

In [77]:
!wc ./vocab.syl.yml

  4165   8332 112760 ./vocab.syl.yml


## For Word Segmented Data

ဒီတခါတော့ word unit အတွက် vocab ဆောက်မယ်။  

In [78]:
%cd ~/exp/nmt/assignment-6/myhi_data/word_segmented/

/home/thant_syn/exp/nmt/assignment-6/myhi_data/word_segmented


In [79]:
!time cat word_test.my  ../test.hi  word_train.my  ../train.hi  word_valid.my  ../valid.hi | marian-vocab > vocab.word.yml

[2026-10-03 06:36:18] Creating vocabulary...
[2026-10-03 06:36:18] [data] Creating vocabulary stdout from stdin
[2026-10-03 06:36:18] Finished

real	0m0.168s
user	0m0.572s
sys	0m0.700s


In [80]:
!head -n 20 ./vocab.word.yml

</s>: 0
<unk>: 1
။: 2
တယ်: 3
𓆑: 4
ပါ: 5
က: 6
ကို: 7
ကျွန်မ: 8
ခဲ့: 9
နေ: 10
ဘူး: 11
𓆑𓈙: 12
𓄿: 13
ဒါ: 14
𓂨𓀀𓎡: 15
𓆑𓈎: 16
လား: 17
𓆑𓉔: 18
သူ: 19


In [81]:
!tail -n 20 ./vocab.word.yml

𓐍𓇋𓂝𓈙𓂝: 5468
𓐍𓇋𓅓𓂝𓏭𓈎: 5469
𓐍𓇋𓈚𓀀𓂋𓂝𓈎𓂝: 5470
𓐍𓇋𓎜𓀀𓍿𓆒𓀀𓎡𓈙: 5471
𓐍𓋴𓄿𓎛𓃀𓂝𓈙: 5472
𓐍𓋴𓏭𓊪𓆑𓇌𓈙: 5473
𓐍𓎡𓈎𓂋𓄿𓏏: 5474
𓐍𓎡𓈎𓇌𓄿𓈎𓂝𓇌: 5475
𓐍𓎡𓈎𓋴𓄿𓃀𓂝: 5476
𓐍𓎡𓈎𓋴𓄿𓎛𓈖: 5477
𓐍𓎡𓈎𓏏𓆑𓈎𓈎𓂝𓇌: 5478
𓐍𓎼𓂝𓋴𓇌𓐍: 5479
𓐍𓎼𓂝𓋴𓋴𓂝𓄿𓏭𓈎𓆑𓎛𓃀: 5480
𓐍𓎼𓂝𓋴𓏏𓂝𓆑𓃀𓊪𓈎: 5481
𓐍𓎼𓂝𓎛: 5482
𓐍𓏏𓂋: 5483
𓐍𓏏𓂝𓈙: 5484
𓐍𓏏𓎛𓂝𓋴: 5485
𓐍𓏏𓎛𓂝𓋴𓈙: 5486
𓐍𓏭𓏭𓎡𓄡𓆑𓂝𓇌: 5487


အထက်မှာ မြင်ရတဲ့အတိုင်းပဲ ပုဒ်ထီး၊ ပုဒ်မ တွေက စာလုံးတွေနဲ့ ပူးနေတာတွေကို တွေ့ရပါလိမ့်မယ်။ Syllable segmentation မှာတော့ sylbreak tool or sylbreak4all tool နဲ့ ဖြတ်လိုက်ရင် ပြဿနာ မရှိပေမဲ့၊ word unit ဖြတ်ထားတဲ့ ဖိုင်မှာတော့ ဘယ်လို လုပ်ကြမလဲ စဉ်းစားကြဖို့ လိုအပ်ပါတယ်။ မဟုတ်ရင် ရလဒ်ကို လာထိခိုက်မှာမို့လို့။  

နောက်အရေးကြီးတဲ့ တချက်က myContradict ဒေတာမှာက စာကြောင်း တစ်ကြောင်းထက် ပိုတဲ့ စာကြောင်းတွေလည်း ပါဝင်ပါသေးတယ်။ ဥပမာ ။ ။ 

In [83]:
!grep -n "။ " ../word_segmented/word_train.my | head -n 20

609:စောင့် ပါဦး ။ ငါ လာ တော့ မယ် ။


In [85]:
!cat /home/thant_syn/exp/nmt/assignment-6/scripts/clean-punctuation.py

# !/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
clean-punctuation.py
Cleaning punctuation characters for Burmese or Myanmar language.
Python port of clean-punctuation.pl (Ye Kyaw Thu, 22 May 2022)

Usage:
    python3 clean-punctuation.py <input-file> > <output-file>
    cat input.txt | python3 clean-punctuation.py > output.txt
"""

import sys
import re

# Punctuation characters to remove (same set as the Perl script)
PUNCT_RE = re.compile(r'[၊။\-()\[\]/\\!?,"<>]')
MULTI_SPACE_RE = re.compile(r' +')


def clean_line(line: str) -> str:
    """Remove target punctuation, trim, and collapse multiple spaces."""
    line = PUNCT_RE.sub('', line)
    line = line.strip()
    line = MULTI_SPACE_RE.sub(' ', line)
    return line


def main() -> None:
    # Read from file argument, or fall back to stdin
    if len(sys.argv) >= 2:
        fin = open(sys.argv[1], 'r', encoding='utf-8')
        close_fin = True
    else:
        fin = sys.stdin
        close_fin = False

    try:
        for raw 

In [86]:
# Clear Punctuation

# !cat word_all.txt | python ../../scripts/clean-punctuation.py > word_all.txt.clean

# !cat syl_all.txt | python ../../scripts/clean-punctuation.py > syl_all.txt.clean


**Syllable normalization လုပ်လိုက်ရင်တော့ အောက်ပါလိုမျိုး ကိစ္စတွေ ပျောက်သွားလိမ့်မယ်။ သို့သော် syllable normalized training လုပ်ထားရင်၊ online testing လုပ်တဲ့အခါမှာလည်း syllable normalize လုပ်ပြီးမှ model ထဲကို input  လုပ်ရလိမ့်မယ်။**  
 

**Command ပေးတာမှာ အမှားအယွင်းမရှိအောင် ဂရုစိုက်ပါ။ ပြန်စစ်ပါ။**  

---




In [94]:
%pwd

'/home/thant_syn/exp/nmt/assignment-6/myhi_data/word_segmented'

In [90]:
!wc *.my

   500   2782  31823 word_test.my
  8500  47042 533698 word_train.my
  1000   5500  61464 word_valid.my
 10000  55324 626985 total


In [92]:
!wc ../*.hi

   500   1573  24117 ../test.hi
  8500  26835 412919 ../train.hi
  1000   3166  48906 ../valid.hi
 10000  31574 485942 total


In [93]:
!wc ../syllable_segmented/*.my

   500   3691  32732 ../syllable_segmented/syl_test.my
  8500  62718 549374 ../syllable_segmented/syl_train.my
  1000   7255  63219 ../syllable_segmented/syl_valid.my
 10000  73664 645325 total


In [96]:
!tree ../

../
├── syllable_segmented
│   ├── syl_test.my
│   ├── syl_train.my
│   └── syl_valid.my
├── test.hi
├── test.my
├── train.hi
├── train.my
├── valid.hi
├── valid.my
├── vocab
│   ├── vocab.syl.yml
│   └── vocab.word.yml
└── word_segmented
    ├── word_test.my
    ├── word_train.my
    └── word_valid.my

4 directories, 14 files


**Segmentation ကို Word level ရော Syllable level နဲ့ပါ ဖြတ်ခဲ့ပါတယ်။ ဒါပေမဲ့ Syllable level ကို အဓိကထားပြီး စမ်းသပ်ထားပါတယ်။**

## Preparing Bash Scripts for Sequence to Sequence Model


In [104]:
%cd ~/exp/nmt/assignment-6/

/home/thant_syn/exp/nmt/assignment-6


In [17]:
!cat s2s.syl.sh

# ============================================================
# MARIAN NMT TRAINING - SYLLABLE MY -> HI
# Faster CPU configuration
# ============================================================
#!/bin/bash

# ============================================================
# PATHS
# ============================================================

data_path="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
vocab_path="${data_path}/vocab"

src="my"
tgt="hi"

model_folder="/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi"

mkdir -p "${model_folder}"

marian \
  --type s2s \
  --train-sets \
    ${data_path}/syllable_segmented/syl_train.my \
    ${data_path}/train.hi \
  --max-length 120 \
  --valid-sets \
    ${data_path}/syllable_segmented/syl_valid.my \
    ${data_path}/valid.hi \
  --vocabs \
    ${vocab_path}/vocab.syl.yml \
    ${vocab_path}/vocab.syl.yml \
  --model ${model_folder}/model.npz \
  --workspace 500 \
  \
  --enc-depth 1 \
  --enc-type alternating \
  --enc-cell lstm \


**ဒီ architecture က training timeကြာလို့ parameter ပြင်ထားပြီး updated shell scripte နဲ့ run ကြည့်ထားပါတယ်။**

## S2S Modelling (Myanmar --> Hiero)

**Train Syllable Unit Data**

In [8]:
# !chmod +x s2s.syl.updated.sh

In [9]:
!cat s2s.syl.updated.sh

#!/bin/bash

data_path="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
vocab_path="${data_path}/vocab"

src="my"
tgt="hi"

model_folder="/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi"

mkdir -p "${model_folder}"

marian \
  --type s2s \
  --train-sets \
    "${data_path}/syllable_segmented/syl_train.my" \
    "${data_path}/train.hi" \
  --max-length 50 \
  --valid-sets \
    "${data_path}/syllable_segmented/syl_valid.my" \
    "${data_path}/valid.hi" \
  --vocabs \
    "${vocab_path}/vocab.syl.yml" \
    "${vocab_path}/vocab.syl.yml" \
  --model "${model_folder}/model.npz" \
  --dim-emb 128 \
  --dim-rnn 128 \
  --enc-type bidirectional \
  --enc-cell gru \
  --enc-depth 1 \
  --dec-cell gru \
  --dec-depth 1 \
  --dropout-rnn 0.1 \
  --dropout-src 0.1 \
  --dropout-trg 0.1 \
  --mini-batch 64 \
  --learn-rate 0.0005 \
  --valid-freq 1000 \
  --save-freq 1000 \
  --disp-freq 500 \
  --valid-metrics ce-mean-words bleu \
  --early-stopping 5 \
  --keep-best \
  --seed 42 \
  

In [ ]:
!time ./s2s.syl.updated.sh

[2026-10-03 08:50:37] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-03 08:50:37] [marian] Running on PU25-003 as process 7351 with command line:
[2026-10-03 08:50:37] [marian] marian --type s2s --train-sets /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_train.my /home/thant_syn/exp/nmt/assignment-6/myhi_data/train.hi --max-length 50 --valid-sets /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_valid.my /home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.hi --vocabs /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml --model /home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi/model.npz --dim-emb 128 --dim-rnn 128 --enc-type bidirectional --enc-cell gru --enc-depth 1 --dec-cell gru --dec-depth 1 --dropout-rnn 0.1 --dropout-src 0.1 --dropout-trg 0.1 --mini-batch 64 --learn-rate 0.0005 --valid-freq 1000 --save-freq 1000 --disp-freq 500 -

In [8]:
# Testing

%cd model.s2s.syl.my-hi/

/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi


In [26]:
!cat test-eval.sh

#!/bin/bash

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
MODEL_DIR="/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi"

VOCAB="${DATA}/vocab/vocab.syl.yml"
TEST_SRC="${DATA}/syllable_segmented/syl_test.my"
TEST_TGT="${DATA}/test.hi"

> eval-result.txt

# Best BLEU model
MODEL="./model.npz.best-bleu.npz"
HYP="./hyp.best-bleu.hi"

marian-decoder \
    -m "${MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    --devices 0 \
    --output "${HYP}" \
    < "${TEST_SRC}"

echo "Evaluation with ${HYP}, Best BLEU model:" >> eval-result.txt

perl /home/ye/tool/mosesbin/ubuntu-17.04/moses/scripts/generic/multi-bleu.perl \
    "${TEST_TGT}" \
    < "${HYP}" >> eval-result.txt


# Best CE model
MODEL="./model.npz.best-ce-mean-words.npz"
HYP="./hyp.best-ce-mean-words.hi"

marian-decoder \
    -m "${MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    --devices 0 \
    --output "${HYP}" \
    < "${TEST_SRC}"

echo "Evaluation with ${HYP}, Best CE model:" >> eval-result.txt

perl /home/ye/tool/mosesb

In [27]:
# !chmod +x test-eval.sh

In [28]:
!time ./test-eval.sh

[2026-10-03 09:48:09] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-03 09:48:09] [marian] Running on PU25-003 as process 8325 with command line:
[2026-10-03 09:48:09] [marian] marian-decoder -m ./model.npz.best-bleu.npz -v /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml --devices 0 --output ./hyp.best-bleu.hi
[2026-10-03 09:48:09] [config] alignment: ""
[2026-10-03 09:48:09] [config] allow-special: false
[2026-10-03 09:48:09] [config] allow-unk: false
[2026-10-03 09:48:09] [config] authors: false
[2026-10-03 09:48:09] [config] beam-size: 12
[2026-10-03 09:48:09] [config] bert-class-symbol: "[CLS]"
[2026-10-03 09:48:09] [config] bert-mask-symbol: "[MASK]"
[2026-10-03 09:48:09] [config] bert-masking-fraction: 0.15
[2026-10-03 09:48:09] [config] bert-sep-symbol: "[SEP]"
[2026-10-03 09:48:09] [config] bert-train-type-embeddings: true
[2026-10-03 09:48:09] [config] bert-type-vocab-si

In [29]:
!head test.hyp.hi

𓂨𓀀𓎡𓋴 𓂋𓐍𓎼𓂝 𓆑𓈙 𓅱𓄿𓈎
𓆑 𓊪𓄿𓎼𓂝 𓎜𓀀 𓅲𓀀𓐍𓇌
𓎜𓀀 𓐍𓎛𓂝 𓈙𓄿𓏏 𓉔𓂝
𓂨𓀀𓎡𓋴𓂝 𓏭𓋴𓄿𓅱𓈎𓂧
𓈎𓊪𓂝 𓈎𓂝𓄿𓉔 𓏏𓄿𓈙 𓂝𓄿𓋴𓂋𓂧
𓂝𓎛𓅓𓐍𓂧 𓂨𓀀𓎡𓋴 𓈎𓋴𓆑𓄡
𓏏𓂝 𓇌𓆑𓇌 𓆑𓈎 𓄿𓂋𓂋
𓈎𓊪𓂝 𓋴𓐍𓐍𓉔 𓆑𓈙 𓏮𓀀𓂋𓇌
𓈙𓊪𓂝 𓂋𓆑𓈖𓂝𓇌 𓈎𓊪𓄿𓈎
𓈎𓊪𓄿𓈎𓈙 𓄿𓂋𓂋


In [30]:
!cat eval-result.txt

Evaluation with ./hyp.best-bleu.hi, Best BLEU model:
Evaluation with ./hyp.best-ce-mean-words.hi, Best CE model:
Evaluation with ./hyp.final.hi, Final model:
BLEU = 24.04, 50.6/27.7/17.9/14.8 (BP=0.974, ratio=0.974, hyp_len=1532, ref_len=1573)


---

## S2S Modeling (Heiro --> Myanmar)

**Training with syllable segmentation**

In [6]:
!head -20 /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml

</s>: 0
<unk>: 1
။: 2
တယ်: 3
မ: 4
ပါ: 5
က: 6
𓆑: 7
ကျွန်: 8
ကို: 9
အ: 10
သူ: 11
ခဲ့: 12
နေ: 13
ဒါ: 14
ဘူး: 15
𓆑𓈙: 16
𓄿: 17
𓂨𓀀𓎡: 18
လား: 19


In [7]:
!tail -20 /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml

𓐍𓇋𓂝𓈙𓂝: 4146
𓐍𓇋𓅓𓂝𓏭𓈎: 4147
𓐍𓇋𓈚𓀀𓂋𓂝𓈎𓂝: 4148
𓐍𓇋𓎜𓀀𓍿𓆒𓀀𓎡𓈙: 4149
𓐍𓋴𓄿𓎛𓃀𓂝𓈙: 4150
𓐍𓋴𓏭𓊪𓆑𓇌𓈙: 4151
𓐍𓎡𓈎𓂋𓄿𓏏: 4152
𓐍𓎡𓈎𓇌𓄿𓈎𓂝𓇌: 4153
𓐍𓎡𓈎𓋴𓄿𓃀𓂝: 4154
𓐍𓎡𓈎𓋴𓄿𓎛𓈖: 4155
𓐍𓎡𓈎𓏏𓆑𓈎𓈎𓂝𓇌: 4156
𓐍𓎼𓂝𓋴𓇌𓐍: 4157
𓐍𓎼𓂝𓋴𓋴𓂝𓄿𓏭𓈎𓆑𓎛𓃀: 4158
𓐍𓎼𓂝𓋴𓏏𓂝𓆑𓃀𓊪𓈎: 4159
𓐍𓎼𓂝𓎛: 4160
𓐍𓏏𓂋: 4161
𓐍𓏏𓂝𓈙: 4162
𓐍𓏏𓎛𓂝𓋴: 4163
𓐍𓏏𓎛𓂝𓋴𓈙: 4164
𓐍𓏭𓏭𓎡𓄡𓆑𓂝𓇌: 4165

In [10]:
!cat s2s.syl.hi-my.sh

#!/bin/bash

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
VOCAB="${DATA}/vocab"

SRC="hi"
TGT="my"

MODEL_DIR="/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.hi-my"

mkdir -p "${MODEL_DIR}"

marian \
  --type s2s \
  --train-sets \
    "${DATA}/train.hi" \
    "${DATA}/syllable_segmented/syl_train.my" \
  --max-length 50 \
  --valid-sets \
    "${DATA}/valid.hi" \
    "${DATA}/syllable_segmented/syl_valid.my" \
  --vocabs \
    "${VOCAB}/vocab.syl.yml" \
    "${VOCAB}/vocab.syl.yml" \
  --model "${MODEL_DIR}/model.npz" \
  --dim-emb 128 \
  --dim-rnn 128 \
  --enc-type bidirectional \
  --enc-cell gru \
  --enc-depth 1 \
  --dec-cell gru \
  --dec-depth 1 \
  --dropout-rnn 0.1 \
  --dropout-src 0.1 \
  --dropout-trg 0.1 \
  --mini-batch 64 \
  --learn-rate 0.0005 \
  --valid-freq 1000 \
  --save-freq 1000 \
  --disp-freq 500 \
  --valid-metrics ce-mean-words bleu \
  --early-stopping 5 \
  --keep-best \
  --seed 42 \
  --log "${MODEL_DIR}/train.log" \
  --valid-log "${MODE

In [18]:
# !chmod +x s2s.syl.hi-my.sh

In [23]:
!time ./s2s.syl.hi-my.sh

[2026-10-03 10:07:02] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-03 10:07:02] [marian] Running on PU25-003 as process 8740 with command line:
[2026-10-03 10:07:02] [marian] marian --type s2s --train-sets /home/thant_syn/exp/nmt/assignment-6/myhi_data/train.hi /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_train.my --max-length 50 --valid-sets /home/thant_syn/exp/nmt/assignment-6/myhi_data/valid.hi /home/thant_syn/exp/nmt/assignment-6/myhi_data/syllable_segmented/syl_valid.my --vocabs /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml --model /home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.hi-my/model.npz --dim-emb 128 --dim-rnn 128 --enc-type bidirectional --enc-cell gru --enc-depth 1 --dec-cell gru --dec-depth 1 --dropout-rnn 0.1 --dropout-src 0.1 --dropout-trg 0.1 --mini-batch 64 --learn-rate 0.0005 --valid-freq 1000 --save-freq 1000 --disp-freq 500 -

In [24]:
%cd model.s2s.syl.hi-my/

/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.hi-my


In [25]:
%ls

model.npz                                     model.npz.optimizer.npz
model.npz.best-bleu.npz                       model.npz.progress.yml
model.npz.best-bleu.npz.decoder.yml           model.npz.yml
model.npz.best-ce-mean-words.npz              train.log
model.npz.best-ce-mean-words.npz.decoder.yml  valid.log
model.npz.decoder.yml


In [26]:
## Testing

In [27]:
!chmod +x test-eval.hi-my.sh

In [28]:
!time ./test-eval.hi-my.sh

[2026-10-03 11:54:20] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-03 11:54:20] [marian] Running on PU25-003 as process 10095 with command line:
[2026-10-03 11:54:20] [marian] marian-decoder -m ./model.npz.best-bleu.npz -v /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml /home/thant_syn/exp/nmt/assignment-6/myhi_data/vocab/vocab.syl.yml --devices 0 --output ./hyp.best-bleu.my
[2026-10-03 11:54:20] [config] alignment: ""
[2026-10-03 11:54:20] [config] allow-special: false
[2026-10-03 11:54:20] [config] allow-unk: false
[2026-10-03 11:54:20] [config] authors: false
[2026-10-03 11:54:20] [config] beam-size: 12
[2026-10-03 11:54:20] [config] bert-class-symbol: "[CLS]"
[2026-10-03 11:54:20] [config] bert-mask-symbol: "[MASK]"
[2026-10-03 11:54:20] [config] bert-masking-fraction: 0.15
[2026-10-03 11:54:20] [config] bert-sep-symbol: "[SEP]"
[2026-10-03 11:54:20] [config] bert-train-type-embeddings: true
[2026-10-03 11:54:20] [config] bert-type-vocab-s

In [33]:
!head hyp.best-bleu.my

ချစ် ကြိုက် တယ် ။
ကျွန် မ မှာ အ စား အ စာ မ ရှိ ဘူး ။
ဘယ် သူ မှ ကျွန် မ ကို မ မြင် ခဲ့ ဘူး ။
မင်း စော င့် ကြ ည့် နေ ပြီ
အ ရမ်း စော င့် နေ ခဲ့ တယ် ။
ခ ရီး ကောင်း ပါ
ဘယ် လို ပဲ ဖြစ် ဖြစ် ခဲ့ တယ် ။
အ ခန်း က မှောင် နေ တယ် ။
သူ မ ဟာ လျှောက် နေ တယ် ။
ဒါ ပဲ


In [30]:
!ls

eval-result.txt
hyp.best-bleu.my
hyp.best-ce-mean-words.my
hyp.final.my
model.npz
model.npz.best-bleu.npz
model.npz.best-bleu.npz.decoder.yml
model.npz.best-ce-mean-words.npz
model.npz.best-ce-mean-words.npz.decoder.yml
model.npz.decoder.yml
model.npz.optimizer.npz
model.npz.progress.yml
model.npz.yml
test-eval.hi-my.sh
train.log
valid.log


In [32]:
!cat eval-result.txt

Evaluation with ./hyp.best-bleu.my, Best BLEU model:
BLEU = 47.02, 73.6/56.3/43.3/35.9 (BP=0.933, ratio=0.935, hyp_len=3452, ref_len=3691)
Evaluation with ./hyp.best-ce-mean-words.my, Best CE model:
BLEU = 45.02, 71.0/53.4/40.3/32.5 (BP=0.954, ratio=0.955, hyp_len=3525, ref_len=3691)
Evaluation with ./hyp.final.my, Final model:
BLEU = 47.03, 73.6/56.7/44.0/36.4 (BP=0.925, ratio=0.928, hyp_len=3424, ref_len=3691)



| Model     |      BLEU | 1-gram | 2-gram | 3-gram | 4-gram |
| --------- | --------: | -----: | -----: | -----: | -----: |
| Best BLEU | **47.02** |   73.6 |   56.3 |   43.3 |   35.9 |
| Best CE   | **45.02** |   71.0 |   53.4 |   40.3 |   32.5 |
| Final     | **47.03** |   73.6 |   56.7 |   44.0 |   36.4 |


---

## Transformer Modelling (Myanmar --> Hiero)

Training with syllable segmentation

In [35]:
%cd ..

/home/thant_syn/exp/nmt/assignment-6


**ဆရာအတန်းထဲမှာ run ခဲ့တဲ့ shell script က parameter များတာကြောင့် အခု personal laptop က RAM နည်းတာကြောင့်ရော၊ CPU နဲ့ run ရတာမလို့**

**parameter နည်းနည်းပြင် ရေးပြီး run ထားပါတယ် ဆရာ**

In [6]:
!cat tf.syl.sh

#!/bin/bash

# ==========================================
# Assignment 6
# Lightweight Transformer
# Myanmar -> Hiero
# ==========================================

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
VOCAB="${DATA}/vocab"

SRC="my"
TGT="hi"

MODEL_DIR="/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.my-hi"

mkdir -p "${MODEL_DIR}"

marian \
    --model "${MODEL_DIR}/model.npz" \
    --type transformer \
    --train-sets \
        "${DATA}/syllable_segmented/syl_train.my" \
        "${DATA}/train.hi" \
    --max-length 50 \
    --vocabs \
        "${VOCAB}/vocab.syl.yml" \
        "${VOCAB}/vocab.syl.yml" \
    --mini-batch-fit -w 8 \
    --maxi-batch 50 \
    --early-stopping 5 \
    --valid-freq 1000 \
    --save-freq 1000 \
    --disp-freq 500 \
    --valid-metrics cross-entropy perplexity bleu \
    --valid-sets \
        "${DATA}/syllable_segmented/syl_valid.my" \
        "${DATA}/valid.hi" \
    --valid-translation-output \
        "${MODEL_DIR}/valid.

**Transformer Architecture:**

- Encoder: 1 layer
- Decoder: 1 layer
- Attention heads: 2
- Embedding: 128
- FFN: 256
- Dropout: 0.1
- Workspace: 8 MB
- Max length: 50
- Beam size: 4

In [7]:
# !chmod +x tf.syl.sh

In [8]:
!time ./tf.syl.sh

[2026-10-03 12:13:04] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-03 12:13:04] [marian] Running on PU25-003 as process 10609 with command line:
[2026-10-03 12:13:04] [marian] marian -c /home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.my-hi/my-hi.config.yml
[2026-10-03 12:13:04] [config] after: 0e
[2026-10-03 12:13:04] [config] after-batches: 0
[2026-10-03 12:13:04] [config] after-epochs: 0
[2026-10-03 12:13:04] [config] all-caps-every: 0
[2026-10-03 12:13:04] [config] allow-unk: false
[2026-10-03 12:13:04] [config] authors: false
[2026-10-03 12:13:04] [config] beam-size: 4
[2026-10-03 12:13:04] [config] bert-class-symbol: "[CLS]"
[2026-10-03 12:13:04] [config] bert-mask-symbol: "[MASK]"
[2026-10-03 12:13:04] [config] bert-masking-fraction: 0.15
[2026-10-03 12:13:04] [config] bert-sep-symbol: "[SEP]"
[2026-10-03 12:13:04] [config] bert-train-type-embeddings: true
[2026-10-03 12:13:04] [config] bert-type-vocab-size: 2
[2026-10-03 12:13:04] [config] buil

In [10]:
%cd model.transformer.syl.my-hi/

/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.my-hi


In [11]:
%ls

model.iter1000.npz              model.iter7000.npz.decoder.yml
model.iter1000.npz.decoder.yml  model.iter8000.npz
model.iter2000.npz              model.iter8000.npz.decoder.yml
model.iter2000.npz.decoder.yml  model.npz
model.iter3000.npz              model.npz.decoder.yml
model.iter3000.npz.decoder.yml  model.npz.optimizer.npz
model.iter4000.npz              model.npz.progress.yml
model.iter4000.npz.decoder.yml  model.npz.yml
model.iter5000.npz              my-hi.config.yml
model.iter5000.npz.decoder.yml  tf.my-hi.log
model.iter6000.npz              train.log
model.iter6000.npz.decoder.yml  valid.log
model.iter7000.npz              valid.my-hi.output


In [16]:
## Testing

!chmod +x test-eval.sh

In [17]:
!cat ./test-eval.sh

#!/bin/bash

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
MODEL_DIR="/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.my-hi"
VOCAB="${DATA}/vocab/vocab.syl.yml"

SRC="${DATA}/syllable_segmented/syl_test.my"
REF="${DATA}/test.hi"

echo "=========================================="
echo "Transformer Myanmar -> Hiero"
echo "=========================================="

# Generate translation
marian-decoder \
    -m "${MODEL_DIR}/model.npz" \
    -v "${VOCAB}" "${VOCAB}" \
    -i "${SRC}" \
    -o "${MODEL_DIR}/hyp.final.hi" \
    --beam-size 4 \
    --normalize 0.6 \
    --max-length 50 \
    --quiet

echo ""
echo "=========================================="
echo "BLEU Score"
echo "=========================================="

/home/thant_syn/mosesdecoder/scripts/generic/multi-bleu.perl \
    "${REF}" < "${MODEL_DIR}/hyp.final.hi"


In [18]:
!time ./test-eval.sh

Transformer Myanmar -> Hiero

BLEU Score
BLEU = 29.13, 57.1/32.6/23.0/17.9 (BP=0.984, ratio=0.984, hyp_len=1548, ref_len=1573)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.

real	0m1.449s
user	0m6.862s
sys	0m9.669s


---

## Transformer Modelling (Hiero --> Myanmar)

Training with syllable segmentation

In [20]:
%cd ..

/home/thant_syn/exp/nmt/assignment-6


In [21]:
%ls

burmese_hiero_mt.ipynb        myhi_data/           scripts/
model.s2s.syl.hi-my/          s2s.syl.hi-my.sh*    tf.syl.hi-my.sh
model.s2s.syl.my-hi/          s2s.syl.sh*          tf.syl.sh*
model.transformer.syl.my-hi/  s2s.syl.updated.sh*


In [26]:
!cat tf.syl.hi-my.sh

#!/bin/bash

# ==========================================
# Assignment 6
# Lightweight Transformer
# Hiero -> Myanmar
# ==========================================

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
VOCAB="${DATA}/vocab"

SRC="hi"
TGT="my"

MODEL_DIR="/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.hi-my"

mkdir -p "${MODEL_DIR}"

marian \
    --model "${MODEL_DIR}/model.npz" \
    --type transformer \
    --train-sets \
        "${DATA}/train.hi" \
        "${DATA}/syllable_segmented/syl_train.my" \
    --max-length 50 \
    --vocabs \
        "${VOCAB}/vocab.syl.yml" \
        "${VOCAB}/vocab.syl.yml" \
    --mini-batch-fit -w 8 \
    --maxi-batch 50 \
    --early-stopping 5 \
    --valid-freq 1000 \
    --save-freq 1000 \
    --disp-freq 500 \
    --valid-metrics cross-entropy perplexity bleu \
    --valid-sets \
        "${DATA}/valid.hi" \
        "${DATA}/syllable_segmented/syl_valid.my" \
    --valid-translation-output \
        "${MODEL_DIR}/valid.

In [27]:
!chmod +x ./tf.syl.hi-my.sh

In [ ]:
!time ./tf.syl.hi-my.sh

[2026-10-03 13:43:55] [marian] Marian v1.12.43 c9f287db 2025-06-23 19:47:45 +0000
[2026-10-03 13:43:55] [marian] Running on PU25-003 as process 11757 with command line:
[2026-10-03 13:43:55] [marian] marian -c /home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.hi-my/hi-my.config.yml
[2026-10-03 13:43:55] [config] after: 0e
[2026-10-03 13:43:55] [config] after-batches: 0
[2026-10-03 13:43:55] [config] after-epochs: 0
[2026-10-03 13:43:55] [config] all-caps-every: 0
[2026-10-03 13:43:55] [config] allow-unk: false
[2026-10-03 13:43:55] [config] authors: false
[2026-10-03 13:43:55] [config] beam-size: 4
[2026-10-03 13:43:55] [config] bert-class-symbol: "[CLS]"
[2026-10-03 13:43:55] [config] bert-mask-symbol: "[MASK]"
[2026-10-03 13:43:55] [config] bert-masking-fraction: 0.15
[2026-10-03 13:43:55] [config] bert-sep-symbol: "[SEP]"
[2026-10-03 13:43:55] [config] bert-train-type-embeddings: true
[2026-10-03 13:43:55] [config] bert-type-vocab-size: 2
[2026-10-03 13:43:55] [config] buil

In [5]:
%ls

burmese_hiero_mt.ipynb        model.transformer.syl.my-hi/  s2s.syl.updated.sh*
model.s2s.syl.hi-my/          myhi_data/                    scripts/
model.s2s.syl.my-hi/          s2s.syl.hi-my.sh*             tf.syl.hi-my.sh*
model.transformer.syl.hi-my/  s2s.syl.sh*                   tf.syl.sh*


In [6]:
%cd model.transformer.syl.hi-my/

/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.hi-my


In [7]:
%ls

hi-my.config.yml                model.iter6000.npz.decoder.yml
model.iter1000.npz              model.iter7000.npz
model.iter1000.npz.decoder.yml  model.iter7000.npz.decoder.yml
model.iter2000.npz              model.npz
model.iter2000.npz.decoder.yml  model.npz.decoder.yml
model.iter3000.npz              model.npz.optimizer.npz
model.iter3000.npz.decoder.yml  model.npz.progress.yml
model.iter4000.npz              model.npz.yml
model.iter4000.npz.decoder.yml  tf.hi-my.log
model.iter5000.npz              train.log
model.iter5000.npz.decoder.yml  valid.hi-my.output
model.iter6000.npz              valid.log


In [9]:
!head valid.hi-my.output

ကျွန် မ အ ရမ်း ကောင်း တယ် လို့ ခံ စား ရ တယ် ။
ကျွန် မ မှာ မြင်း မ ရှိ ဘူး ။
ကျွန် တော် တို့ သူ့ ကို ငို စေ ခဲ့ တယ် ။
နည်း နည်း နီး ကပ် လာ ပါ ။
ကျွန် မ ယုံ ကြည် မှု ရှိ တယ်
ကျွန် မ က သ ကြား လုံး တွေ ကို ကြိုက် တယ် ။
စိတ် မ ကောင်း ဖြစ် မိ တယ် ။
ဒါ က ကောင်း တယ် ။
ပို ကောင်း လာ ပြီ ။
စား ခဲ့ တယ် ။


In [10]:
!cat test-eval.sh

#!/bin/bash

# ==========================================
# Assignment 6
# Lightweight Transformer
# Hiero -> Myanmar
# Testing
# ==========================================

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"
MODEL_DIR="/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.hi-my"
VOCAB="${DATA}/vocab/vocab.syl.yml"

SRC="${DATA}/test.hi"
REF="${DATA}/syllable_segmented/syl_test.my"

echo "=========================================="
echo "Transformer Hiero -> Myanmar"
echo "=========================================="

# Generate translation
marian-decoder \
    -m "${MODEL_DIR}/model.npz" \
    -v "${VOCAB}" "${VOCAB}" \
    -i "${SRC}" \
    -o "${MODEL_DIR}/hyp.final.my" \
    --beam-size 4 \
    --normalize 0.6 \
    --max-length 50 \
    --quiet

echo ""
echo "=========================================="
echo "BLEU Score"
echo "=========================================="

/home/thant_syn/mosesdecoder/scripts/generic/multi-bleu.perl \
    "${REF}" < "${MODEL_DI

In [12]:
!chmod +x test-eval.sh

In [13]:
!time ./test-eval.sh

Transformer Hiero -> Myanmar

BLEU Score
BLEU = 45.23, 71.4/53.5/40.2/31.8 (BP=0.962, ratio=0.963, hyp_len=3555, ref_len=3691)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.

real	0m2.839s
user	0m13.379s
sys	0m19.459s


**Hiero → Myanmar details**
```python
BLEU       = 45.23
1-gram     = 71.4
2-gram     = 53.5
3-gram     = 40.2
4-gram     = 31.8
BP         = 0.962
Ratio      = 0.963
Hyp length = 3555
Ref length = 3691
```

**Comparison between Two Transformer Models**

| Model                   | Direction       |      BLEU |
| ----------------------- | --------------- | --------: |
| Lightweight Transformer | Myanmar → Hiero | **29.13** |
| Lightweight Transformer | Hiero → Myanmar | **45.23** |


The hypothesis/reference length ratio of 0.963 means the generated Myanmar output is somewhat shorter than the reference.

---

### Models Comparison

| Model       | Direction       |      BLEU |
| ----------- | --------------- | --------: |
| S2S         | Myanmar → Hiero | **24.04** |
| Transformer | Myanmar → Hiero | **29.13** |
| S2S         | Hiero → Myanmar | **47.02** |
| Transformer | Hiero → Myanmar | **45.23** |


Transformer model (Hiero → Myanmar) က BLEU score က early stopping မြှင့်လိုက်ရင် ပိုတက်လာနိုင်တယ် လို့ ထင်ပါတယ်ဆရာ။ 

အခုက metrics က သုံးခုဖြစ်နေကို ကြည့်တာကြောင့် ကျန်တဲ့ perplexity နဲ့ cross_entropy တို့က မကျသွားရင် early stopping မှာ count လုပ်သွားတာကြောင့် 

BLEU score တိုးတက်တာကနေ ရပ်သွားတယ်လို့ မြင်မိပါတယ် ဆရာ။

---

## Explore Ensemble Method

### Myanmar --> Heiro

In [17]:
%pwd

'/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.hi-my'

**Pin the model Directories**

```python
S2S:
 /home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi/model.npz

Transformer:
 /home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.my-hi/model.npz
```

In [20]:
%ls

hi-my.config.yml                model.iter6000.npz.decoder.yml
hyp.final.my                    model.iter7000.npz
model.iter1000.npz              model.iter7000.npz.decoder.yml
model.iter1000.npz.decoder.yml  model.npz
model.iter2000.npz              model.npz.decoder.yml
model.iter2000.npz.decoder.yml  model.npz.optimizer.npz
model.iter3000.npz              model.npz.progress.yml
model.iter3000.npz.decoder.yml  model.npz.yml
model.iter4000.npz              test-eval.sh*
model.iter4000.npz.decoder.yml  tf.hi-my.log
model.iter5000.npz              train.log
model.iter5000.npz.decoder.yml  valid.hi-my.output
model.iter6000.npz              valid.log


In [21]:
%cd ../model.ensemble.syl.my-hi/

/home/thant_syn/exp/nmt/assignment-6/model.ensemble.syl.my-hi


In [25]:
!cat ./test-ensemble.sh

#!/bin/bash

# ==========================================
# Assignment 6
# Model Ensemble: S2S + Transformer
# Myanmar -> Hiero
# ==========================================

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"

S2S_MODEL="/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.my-hi/model.npz"
TF_MODEL="/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.my-hi/model.npz"

VOCAB="${DATA}/vocab/vocab.syl.yml"

SRC="${DATA}/syllable_segmented/syl_test.my"
REF="${DATA}/test.hi"

OUT="/home/thant_syn/exp/nmt/assignment-6/model.ensemble.s2s-tf"

mkdir -p "${OUT}"

echo "=========================================="
echo "Model Ensemble: S2S + Transformer"
echo "Myanmar -> Hiero"
echo "=========================================="

marian-decoder \
    -m "${S2S_MODEL}" "${TF_MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    -i "${SRC}" \
    -o "${OUT}/hyp.ensemble.hi" \
    --beam-size 4 \
    --normalize 0.6 \
    --max-length 50 \
    --quiet

echo ""
echo "====================

In [26]:
!chmod +x test-ensemble.sh

In [27]:
!time ./test-ensemble.sh

Model Ensemble: S2S + Transformer
Myanmar -> Hiero

Ensemble BLEU Score
BLEU = 32.50, 58.6/35.1/25.4/22.6 (BP=0.987, ratio=0.987, hyp_len=1552, ref_len=1573)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.

real	0m7.055s
user	0m27.747s
sys	0m43.847s


**BLEU 32.5 ဆိုတော့ ပိုကောင်းလာပါတယ်။**

| Approach                       |      BLEU |
| ------------------------------ | --------: |
| S2S                            |     24.04 |
| Lightweight Transformer        |     29.13 |
| **S2S + Transformer Ensemble** | **32.50** |


> Model Ensemble: An ensemble of the S2S and lightweight Transformer models was evaluated for Myanmar-to-Hiero translation. The S2S model achieved a BLEU score of 24.04, while the lightweight Transformer achieved 29.13. Combining the two models through ensemble decoding resulted in a BLEU score of 32.50 on the test set. This experiment shows that combining predictions from models with different architectures can produce a different translation output and, in this experiment, achieved a higher BLEU score than either individual model.

### Heiro --> Myanmar

In [28]:
%cd ../model.ensemble.syl.hi-my/

/home/thant_syn/exp/nmt/assignment-6/model.ensemble.syl.hi-my


In [29]:
!cat test-ensemble.sh

#!/bin/bash

# ==========================================
# Assignment 6
# Model Ensemble: S2S + Transformer
# Hiero -> Myanmar
# ==========================================

DATA="/home/thant_syn/exp/nmt/assignment-6/myhi_data"

S2S_MODEL="/home/thant_syn/exp/nmt/assignment-6/model.s2s.syl.hi-my/model.npz"
TF_MODEL="/home/thant_syn/exp/nmt/assignment-6/model.transformer.syl.hi-my/model.npz"

VOCAB="${DATA}/vocab/vocab.syl.yml"

SRC="${DATA}/test.hi"
REF="${DATA}/syllable_segmented/syl_test.my"

OUT="/home/thant_syn/exp/nmt/assignment-6/model.ensemble.s2s-tf.hi-my"

mkdir -p "${OUT}"

echo "=========================================="
echo "Model Ensemble: S2S + Transformer"
echo "Hiero -> Myanmar"
echo "=========================================="

marian-decoder \
    -m "${S2S_MODEL}" "${TF_MODEL}" \
    -v "${VOCAB}" "${VOCAB}" \
    -i "${SRC}" \
    -o "${OUT}/hyp.ensemble.my" \
    --beam-size 4 \
    --normalize 0.6 \
    --max-length 50 \
    --quiet

echo ""
echo "==============

In [30]:
!chmod +x ./test-ensemble.sh

In [31]:
!time ./test-ensemble.sh

Model Ensemble: S2S + Transformer
Hiero -> Myanmar

Ensemble BLEU Score
BLEU = 50.06, 74.9/58.6/45.9/38.4 (BP=0.949, ratio=0.951, hyp_len=3509, ref_len=3691)
It is not advisable to publish scores from multi-bleu.perl.  The scores depend on your tokenizer, which is unlikely to be reproducible from your paper or consistent across research groups.  Instead you should detokenize then use mteval-v14.pl, which has a standard tokenization.  Scores from multi-bleu.perl can still be used for internal purposes when you have a consistent tokenizer.

real	0m6.360s
user	0m31.502s
sys	0m44.422s


| Model                          | Myanmar → Hiero | Hiero → Myanmar |
| ------------------------------ | --------------: | --------------: |
| S2S                            |           24.04 |           47.02 |
| Lightweight Transformer        |           29.13 |           45.23 |
| **S2S + Transformer Ensemble** |       **32.50** |       **50.06** |


အကျဉ်းချုပ်အားဖြင့် S2S + Transformer Ensemble သည် Myanmar - Hiero ဘာသာပြန်ခြင်းနှစ်မျိုးစလုံးတွင် individual models တွေထက် BLEU score ပိုမိုမြင့်မားစွာ ရရှိခဲ့ပါတယ်။

---